# 10 — V4: V3 + SpecAugment, và beam search

- **V4** = công thức V3 giữ nguyên (dữ liệu V3, LR 3e-4, 3 epoch, LoRA, seed) + **SpecAugment** khi train: che ~5% khung thời gian
  (đoạn 100 ms) và đôi khi một dải 10 kênh mel. V3 chạm đáy ở bước 1800 rồi đi ngang; SpecAugment là regularization để vượt điểm đó.
  Chỉ đổi một biến nên V4 so V3 cho biết đúng tác dụng của SpecAugment. Eval không bị che.
- **Beam search 5** (V3B, V4B): chỉ giải mã lại, không train. Thường giảm WER so với greedy; tốn thời gian giải mã hơn.

Thứ tự gợi ý: train V4 (~16 giờ) → eval V4 → beam V3B/V4B (mỗi model ~2–3 giờ). Beam V3B chạy được bất cứ lúc nào GPU rảnh.
Mỗi lần chỉ một job GPU. Chạy lại cell train với cùng `RUN_ID` **và cùng `DUTY`** để tiếp tục từ checkpoint gần nhất.
Giữ VS Code mở, cắm sạc, tắt sleep, đóng bớt Chrome/Zalo (bộ nhớ commit của máy chỉ còn ~4 GB trống).


In [1]:
from pathlib import Path
import sys, json
here = Path.cwd().resolve()
PROJECT = next((p for p in [here, *here.parents] if (p/'asr/config.py').is_file()), None)
if PROJECT is None:
    PROJECT = next((p/'ASR' for p in [here, *here.parents] if (p/'ASR/asr/config.py').is_file()), None)
assert PROJECT is not None, 'Mở notebook từ trong workspace dự án'
PYTHON = PROJECT.parent/'.venv-asr/Scripts/python.exe'
assert PYTHON.is_file(), 'Chạy ASR/local/setup.ps1 trước'
sys.path.insert(0, str(PROJECT/'local'))
from notebook_process import run_cli as _run_cli
def run_cli(arguments):
    return _run_cli(PROJECT, PYTHON, arguments)
RUNNER = PROJECT/'scripts/local_run.py'
RUNS = PROJECT/'local/runs'

EXP = 'V4_phowhisper_small_lora_vsplit_specaug'
RUN_ID = 'local_v4'
DUTY = '1.0'
RUN = RUNS/EXP/RUN_ID
FINAL = RUN/'final'
V3_RUN = RUNS/'V3_phowhisper_small_lora_vsplit/local_v3'
assert (PROJECT/'data/ViMedCSS_vsplit/train.parquet').is_file(), 'Thiếu dữ liệu V3: chạy notebook 09 phần Dữ liệu V3'

def gpu_job_running():
    """PID của job GPU local đang chạy (giữ local/gpu.lock), hoặc None."""
    import psutil
    lock = PROJECT/'local/gpu.lock'
    if not lock.exists():
        return None
    pid = int(lock.read_text())
    return pid if psutil.pid_exists(pid) else None

def latest_checkpoint(run):
    ckpts = sorted((run/'checkpoints').glob('checkpoint-*'), key=lambda p: int(p.name.split('-')[1]))
    return ckpts[-1] if ckpts else None

print('Run:', RUN)
print('Checkpoint gần nhất:', latest_checkpoint(RUN) or 'chưa có (train sẽ bắt đầu từ bước 0)')
print('Job GPU đang chạy:', gpu_job_running() or 'không')


Run: D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\runs\V4_phowhisper_small_lora_vsplit_specaug\local_v4
Checkpoint gần nhất: chưa có (train sẽ bắt đầu từ bước 0)
Job GPU đang chạy: không


In [2]:
run_cli([RUNNER, 'check', EXP, '--run-id', RUN_ID])


[START] "D:\1.FPT_FALL26\DPL - Deep Learning\.venv-asr\Scripts\python.exe" -u "D:\1.FPT_FALL26\DPL - Deep Learning\ASR\scripts\local_run.py" check V4_phowhisper_small_lora_vsplit_specaug --run-id local_v4
[LOG] D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\logs\20261025_232435_182658.log
[INIT] mode=check experiment=V4_phowhisper_small_lora_vsplit_specaug; importing PyTorch/ASR libraries...
{
  "python": "D:\\1.FPT_FALL26\\DPL - Deep Learning\\.venv-asr\\Scripts\\python.exe",
  "cuda": true,
  "versions": {
    "torch": "2.6.0+cu124",
    "transformers": "4.46.3",
    "peft": "0.13.2",
    "accelerate": "1.1.1",
    "datasets": "3.1.0"
  },
  "missing_data": [],
  "profile": {
    "total_gpu_budget_gb": 4.5,
    "gpu_duty_cycle": 0.7,
    "host_ram_budget_gb": 0,
    "save_steps": 100,
    "checkpoint_interval_seconds": 600,
    "save_total_limit": 2,
    "batch_size": 1,
    "grad_accum": 32,
    "num_workers": 0
  },
  "local_v1_final": false,
  "system_ram_total_gb": 16.989728768,
 

## 1. Train V4

Smoke 2 bước trước (đã chạy đạt một lần thì cell báo đã hoàn tất), rồi train đầy đủ: ~2800 bước ở DUTY 1,0 → khoảng 16 giờ.


In [3]:
run_cli([RUNNER, 'train', EXP, '--run-id', 'smoke_v4', '--smoke', '--duty-cycle', DUTY])


[START] "D:\1.FPT_FALL26\DPL - Deep Learning\.venv-asr\Scripts\python.exe" -u "D:\1.FPT_FALL26\DPL - Deep Learning\ASR\scripts\local_run.py" train V4_phowhisper_small_lora_vsplit_specaug --run-id smoke_v4 --smoke --duty-cycle 1.0
[LOG] D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\logs\20261025_232443_034675.log
[INIT] mode=train experiment=V4_phowhisper_small_lora_vsplit_specaug; importing PyTorch/ASR libraries...
{
  "python": "D:\\1.FPT_FALL26\\DPL - Deep Learning\\.venv-asr\\Scripts\\python.exe",
  "cuda": true,
  "versions": {
    "torch": "2.6.0+cu124",
    "transformers": "4.46.3",
    "peft": "0.13.2",
    "accelerate": "1.1.1",
    "datasets": "3.1.0"
  },
  "missing_data": [],
  "profile": {
    "total_gpu_budget_gb": 4.5,
    "gpu_duty_cycle": 1.0,
    "host_ram_budget_gb": 0,
    "save_steps": 100,
    "checkpoint_interval_seconds": 600,
    "save_total_limit": 2,
    "batch_size": 1,
    "grad_accum": 32,
    "num_workers": 0
  },
  "local_v1_final": false,
  "system_ram_t

In [4]:
pid = gpu_job_running()
if pid:
    print(f'Đang có job GPU khác (PID {pid}); không khởi chạy thêm. Đợi job đó xong hoặc dừng nó trước.')
else:
    print('Train V4: bắt đầu mới hoặc tiếp tục từ', latest_checkpoint(RUN) or 'bước 0', flush=True)
    run_cli([RUNNER, 'train', EXP, '--run-id', RUN_ID, '--duty-cycle', DUTY])
    print('Adapter tốt nhất:', FINAL if (FINAL/'adapter_model.safetensors').is_file() else 'chưa có')


Train V4: bắt đầu mới hoặc tiếp tục từ bước 0
[START] "D:\1.FPT_FALL26\DPL - Deep Learning\.venv-asr\Scripts\python.exe" -u "D:\1.FPT_FALL26\DPL - Deep Learning\ASR\scripts\local_run.py" train V4_phowhisper_small_lora_vsplit_specaug --run-id local_v4 --duty-cycle 1.0
[LOG] D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\logs\20261025_232509_318288.log
[INIT] mode=train experiment=V4_phowhisper_small_lora_vsplit_specaug; importing PyTorch/ASR libraries...
{
  "python": "D:\\1.FPT_FALL26\\DPL - Deep Learning\\.venv-asr\\Scripts\\python.exe",
  "cuda": true,
  "versions": {
    "torch": "2.6.0+cu124",
    "transformers": "4.46.3",
    "peft": "0.13.2",
    "accelerate": "1.1.1",
    "datasets": "3.1.0"
  },
  "missing_data": [],
  "profile": {
    "total_gpu_budget_gb": 4.5,
    "gpu_duty_cycle": 1.0,
    "host_ram_budget_gb": 0,
    "save_steps": 100,
    "checkpoint_interval_seconds": 600,
    "save_total_limit": 2,
    "batch_size": 1,
    "grad_accum": 32,
    "num_workers": 0
  },
  "l

## Theo dõi: V4 so với V3 trên cùng DEV-subset

Chỉ đọc file. Cùng dữ liệu, cùng 600 clip DEV-subset, nên hai đường `eval_wer` so trực tiếp được (V3 tốt nhất 0,1199 ở bước 1800).


In [5]:
def eval_curve(run):
    ckpt = latest_checkpoint(run)
    if ckpt is None:
        return {}
    state = json.loads((ckpt/'trainer_state.json').read_text(encoding='utf-8'))
    return {e['step']: e['eval_wer'] for e in state['log_history'] if 'eval_wer' in e}

v3, v4 = eval_curve(V3_RUN), eval_curve(RUN)
print('bước   V3_wer  V4_wer')
for step in sorted(v4):
    print(f"{step:5d}  {v3.get(step, float('nan')):.4f}  {v4[step]:.4f}")
if v4:
    print('Tốt nhất: V3', round(min(v3.values()), 4), '| V4', round(min(v4.values()), 4))
logs = sorted((PROJECT/'local/logs').glob('*.log'))
mine = [p for p in logs if f'mode=train experiment={EXP}' in p.read_text(encoding='utf-8', errors='replace')[:300]]
if mine:
    lines = mine[-1].read_text(encoding='utf-8', errors='replace').replace('\r', '\n').splitlines()
    progress = [l for l in lines if l.startswith('[TRAIN] ') and 'microbatch' not in l]
    print('\nLog:', mine[-1])
    print(*(progress[-3:] or lines[-5:]), sep='\n')


bước   V3_wer  V4_wer
  100  0.1534  0.1559
  200  0.1540  0.1642
  300  0.1470  0.1553
  400  0.1458  0.1441
  500  0.1374  0.1296
  600  0.1390  0.1425
  700  0.1288  0.1277
  800  0.1439  0.1307
  900  0.1307  0.1296
 1000  0.1308  0.1312
 1100  0.1298  0.1268
 1200  0.1261  0.1266
 1300  0.1296  0.1310
 1400  0.1201  0.1230
 1500  0.1298  0.1224
 1600  0.1231  0.1250
 1700  0.1228  0.1244
 1800  0.1199  0.1205
 1900  0.1242  0.1214
 2000  0.1275  0.1246
 2100  0.1225  0.1230
 2200  0.1256  0.1220
 2300  0.1234  0.1231
 2400  0.1228  0.1211
 2500  0.1246  0.1215
 2600  0.1249  0.1191
 2700  0.1239  0.1217
 2800  0.1234  0.1221
Tốt nhất: V3 0.1199 | V4 0.1191

Log: D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\logs\20261025_232509_318288.log
[TRAIN] 2800/2802 (99.9%) | epoch=2.995 | ETA~0.5 min | RAM=5.03 GB | VRAM(total)=3.05 GB
[TRAIN] 2801/2802 (100.0%) | epoch=2.996 | ETA~0.3 min | RAM=5.03 GB | VRAM(total)=2.78 GB
[TRAIN] 2802/2802 (100.0%) | epoch=2.997 | ETA~0.0 min | RAM=5.01

## 2. Đánh giá V4 trên DEV V3 (greedy, có guard)


In [6]:
if not (FINAL/'adapter_model.safetensors').is_file():
    print('Chưa có', FINAL, '- train V4 xong trước.')
else:
    run_cli([RUNNER, 'eval', EXP, '--run-id', 'smoke_local_v4', '--max-per-source', '2', '--checkpoint', FINAL,
             '--duty-cycle', DUTY])


[START] "D:\1.FPT_FALL26\DPL - Deep Learning\.venv-asr\Scripts\python.exe" -u "D:\1.FPT_FALL26\DPL - Deep Learning\ASR\scripts\local_run.py" eval V4_phowhisper_small_lora_vsplit_specaug --run-id smoke_local_v4 --max-per-source 2 --checkpoint "D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\runs\V4_phowhisper_small_lora_vsplit_specaug\local_v4\final" --duty-cycle 1.0
[LOG] D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\logs\20261026_114828_271911.log
[INIT] mode=eval experiment=V4_phowhisper_small_lora_vsplit_specaug; importing PyTorch/ASR libraries...
[RUNNING] 11s elapsed; waiting for the next model/data update.
{
  "python": "D:\\1.FPT_FALL26\\DPL - Deep Learning\\.venv-asr\\Scripts\\python.exe",
  "cuda": true,
  "versions": {
    "torch": "2.6.0+cu124",
    "transformers": "4.46.3",
    "peft": "0.13.2",
    "accelerate": "1.1.1",
    "datasets": "3.1.0"
  },
  "missing_data": [],
  "profile": {
    "total_gpu_budget_gb": 4.5,
    "gpu_duty_cycle": 1.0,
    "host_ram_budget_gb": 0,
  

In [7]:
if not (FINAL/'adapter_model.safetensors').is_file():
    print('Chưa có', FINAL, '- train V4 xong trước.')
else:
    run_cli([RUNNER, 'eval', EXP, '--run-id', 'dev_local_v4', '--checkpoint', FINAL, '--duty-cycle', DUTY])


[START] "D:\1.FPT_FALL26\DPL - Deep Learning\.venv-asr\Scripts\python.exe" -u "D:\1.FPT_FALL26\DPL - Deep Learning\ASR\scripts\local_run.py" eval V4_phowhisper_small_lora_vsplit_specaug --run-id dev_local_v4 --checkpoint "D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\runs\V4_phowhisper_small_lora_vsplit_specaug\local_v4\final" --duty-cycle 1.0
[LOG] D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\logs\20261026_114906_712726.log
[INIT] mode=eval experiment=V4_phowhisper_small_lora_vsplit_specaug; importing PyTorch/ASR libraries...
{
  "python": "D:\\1.FPT_FALL26\\DPL - Deep Learning\\.venv-asr\\Scripts\\python.exe",
  "cuda": true,
  "versions": {
    "torch": "2.6.0+cu124",
    "transformers": "4.46.3",
    "peft": "0.13.2",
    "accelerate": "1.1.1",
    "datasets": "3.1.0"
  },
  "missing_data": [],
  "profile": {
    "total_gpu_budget_gb": 4.5,
    "gpu_duty_cycle": 1.0,
    "host_ram_budget_gb": 0,
    "save_steps": 100,
    "checkpoint_interval_seconds": 600,
    "save_total_limit":

## 3. Beam search 5 (chỉ giải mã lại)

V3B chạy được ngay khi GPU rảnh; V4B cần V4 train xong. Mỗi model ~2–3 giờ.


In [8]:
run_cli([RUNNER, 'eval', 'V3B_phowhisper_small_lora_vsplit_beam5', '--run-id', 'dev_beam5', '--duty-cycle', DUTY])


[START] "D:\1.FPT_FALL26\DPL - Deep Learning\.venv-asr\Scripts\python.exe" -u "D:\1.FPT_FALL26\DPL - Deep Learning\ASR\scripts\local_run.py" eval V3B_phowhisper_small_lora_vsplit_beam5 --run-id dev_beam5 --duty-cycle 1.0
[LOG] D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\logs\20261026_124034_280485.log
[INIT] mode=eval experiment=V3B_phowhisper_small_lora_vsplit_beam5; importing PyTorch/ASR libraries...
{
  "python": "D:\\1.FPT_FALL26\\DPL - Deep Learning\\.venv-asr\\Scripts\\python.exe",
  "cuda": true,
  "versions": {
    "torch": "2.6.0+cu124",
    "transformers": "4.46.3",
    "peft": "0.13.2",
    "accelerate": "1.1.1",
    "datasets": "3.1.0"
  },
  "missing_data": [],
  "profile": {
    "total_gpu_budget_gb": 4.5,
    "gpu_duty_cycle": 1.0,
    "host_ram_budget_gb": 0,
    "save_steps": 100,
    "checkpoint_interval_seconds": 600,
    "save_total_limit": 2,
    "batch_size": 1,
    "grad_accum": 32,
    "num_workers": 0
  },
  "local_v1_final": false,
  "system_ram_total_gb": 1

In [10]:
if not (FINAL/'adapter_model.safetensors').is_file():
    print('Chưa có', FINAL, '- train V4 xong trước.')
else:
    run_cli([RUNNER, 'eval', 'V4B_phowhisper_small_lora_vsplit_specaug_beam5', '--run-id', 'dev_beam5', '--duty-cycle', DUTY])


[START] "D:\1.FPT_FALL26\DPL - Deep Learning\.venv-asr\Scripts\python.exe" -u "D:\1.FPT_FALL26\DPL - Deep Learning\ASR\scripts\local_run.py" eval V4B_phowhisper_small_lora_vsplit_specaug_beam5 --run-id dev_beam5 --duty-cycle 1.0
[LOG] D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\logs\20261026_214449_937656.log
[INIT] mode=eval experiment=V4B_phowhisper_small_lora_vsplit_specaug_beam5; importing PyTorch/ASR libraries...
[RUNNING] 11s elapsed; waiting for the next model/data update.
{
  "python": "D:\\1.FPT_FALL26\\DPL - Deep Learning\\.venv-asr\\Scripts\\python.exe",
  "cuda": true,
  "versions": {
    "torch": "2.6.0+cu124",
    "transformers": "4.46.3",
    "peft": "0.13.2",
    "accelerate": "1.1.1",
    "datasets": "3.1.0"
  },
  "missing_data": [],
  "profile": {
    "total_gpu_budget_gb": 4.5,
    "gpu_duty_cycle": 1.0,
    "host_ram_budget_gb": 0,
    "save_steps": 100,
    "checkpoint_interval_seconds": 600,
    "save_total_limit": 2,
    "batch_size": 1,
    "grad_accum": 32,


## So sánh trên DEV V3


In [11]:
import pandas as pd
rows = {}
for name, path in [('B0V', RUNS/'B0V_phowhisper_small_zeroshot_vsplit/dev_guard'),
                   ('V3', RUNS/'V3_phowhisper_small_lora_vsplit/dev_local_v3'),
                   ('V3B beam5', RUNS/'V3B_phowhisper_small_lora_vsplit_beam5/dev_beam5'),
                   ('V4', RUNS/EXP/'dev_local_v4'),
                   ('V4B beam5', RUNS/'V4B_phowhisper_small_lora_vsplit_specaug_beam5/dev_beam5')]:
    path = path/'eval/dev/metrics.json'
    if path.is_file():
        m = json.loads(path.read_text(encoding='utf-8'))
        rows[name] = {'overall': 100*m['overall']['wer'],
                      **{s: 100*v['wer'] for s, v in m['by_source'].items()},
                      'term recall': 100*m['overall'].get('cs_term_recall', float('nan')),
                      'RTF': m['rtf_decode_only']}
display(pd.DataFrame(rows).round(3).rename_axis('WER % (DEV V3, guarded decoding)'))


,B0V,V3,V3B beam5,V4,V4B beam5
"WER % (DEV V3, guarded decoding)",,,,,
overall,24.829,18.264,17.437,17.965,17.257
FLEURS,10.886,8.635,8.373,8.489,8.431
VIVOS,0.420,2.052,1.855,2.275,2.275
ViMedCSS_vsplit,29.051,16.595,15.844,16.572,15.399
VietMed,25.865,21.635,20.643,21.120,20.547
term recall,9.051,60.112,64.107,62.484,66.417
RTF,0.101,0.097,0.142,0.088,0.208
